# NGspice sub circuit for a noise-free DIN A VCVS weighting filter
NGspice doesn't allow you to use the Laplace variable in expressions of controlled sources.
This notebook shows the design of a subcircuit for a noise-free single-input - single-output filter.
The theory is discussed in Chapter 18.4.2 and Figure 18.27.

In [11]:
import SLiCAP as sl 
import sympy as sp
sl.initProject("DIN_A", notebook = True)

Compiling library: SLiCAP.lib.
Compiling library: SLiCAPmodels.lib.


## SLiCAP DIN_A transfer function

In [12]:
DIN_A = sl.DIN_A()
num_dinA, den_dinA = DIN_A.as_numer_denom()
sl.eqn2html("DIN_A", DIN_A)

## Laplace prototype

In [21]:
# We need five coefficients (four different poles and a gain coefficient)
a, b, c, d, e = sp.symbols("a, b, c, d, e")
# And the Fourier and Laplace variables
f = sl.ini.frequency
s = sl.ini.laplace
# The structure of the characteristic poly (the denominator) of the built-in DIN A function is:
den_proto = e * (sp.sqrt(f**2 + a**2)*sp.sqrt(f**2 + b**2)*(f**2 + c**2)*(f**2 + d**2))
sl.eqn2html("DENOM_proto", den_proto)

## Determination of the coefficients
We will determine the numeric values of the coefficients of the characteristic poly of the Laplace prototype by equating these coefficients with those of the characteristic polynomial of the built-in DIN_A function and solving the obtained set of equations.

In [14]:
# We can create a poly of the prototype function after taking the square of it
poly_sq = sp.poly(sp.expand(den_proto**2), f)
# We also create a poly of the square of the denominator of the built-in function
poly_den_dinA_sq = sp.poly(den_dinA**2, f)
# We now need to equate the coefficients of both polys and solve the variables
coeffs_proto    = poly_sq.coeffs()
coeffs_den_dinA = poly_den_dinA_sq.coeffs()
# Construct a list with equations
equations = []
for i in range(len(coeffs_proto)):
    eqn = sp.Eq(coeffs_proto[i], coeffs_den_dinA[i])
    equations.append(eqn)
# Solve the set of equations
solutions = sp.solve(equations, (a, b, c, d, e))
# Select solutions with positive coefficients (as a result of squaring; we got negative as well)
valid = []
for sol in solutions:
    all_positive = True
    for coeff in sol:
        if coeff < 0:
            all_positive = False
            break
    if all_positive:
        valid.append(sol)
# Any resulting solution is OK, we just take the first one and create the characteristic poly
solution = valid[0]
terms =  (1 + sl.ini.laplace/2/sp.pi/solution[0])
terms *= (1 + sl.ini.laplace/2/sp.pi/solution[1])
terms *= (1 + sl.ini.laplace/2/sp.pi/solution[2])**2
terms *= (1 + sl.ini.laplace/2/sp.pi/solution[3])**2
sl.eqn2html("DEN_proto", terms)

In [15]:
# Determine the coefficients of the characteristic poly:
terms = sp.expand(terms)
denom_dinA = sp.poly(terms, s)
coeffs = denom_dinA.coeffs()
# The numerator has just one coefficient (not automatically detected): 4-th order
din_A_laplace = sl.ini.laplace**4 / terms
# Adjust to unity gain to unity at 1 kHz:
mag_1kHz      = sp.Abs(din_A_laplace.subs(sl.ini.laplace, 2*sp.pi*sp.I*1000))
din_A_laplace = sp.N(din_A_laplace/mag_1kHz)
sl.eqn2html("DEN_poly_s", din_A_laplace)

In [16]:
print("\n*Numerator (copy this in the KiCAD schematic diagram):")
print(".param b_4=", str(sp.N(1/mag_1kHz)))
coeffs.reverse()
print("\n*Denominator (copy this in the KiCAD schematic diagram):")
for i in range(len(coeffs)):
    print(".param a_{} = {}".format(str(i), str(sp.N(coeffs[i]))))


*Numerator (copy this in the KiCAD schematic diagram):
.param b_4= 2.39511182980200e-11

*Denominator (copy this in the KiCAD schematic diagram):
.param a_0 = 1.00000000000000
.param a_1 = 0.0171714881392805
.param a_2 = 8.66240986229945e-5
.param a_3 = 1.08260399772761e-7
.param a_4 = 2.18072514015132e-11
.param a_5 = 5.14693104782209e-16
.param a_6 = 3.24101285871316e-21


## Checking of the results

In [17]:
# Create subcircuit; See network theory Section 18.4.2, Figure 18.27 
# The coefficient parameters have been entered manually
sl.makeCircuit("kicad/A1/DIN_A_SPICE.kicad_sch", language="SPICE")
sl.img2html("DIN_A_SPICE.svg", 1000)

Creating netlist of kicad/A1/DIN_A_SPICE.kicad_sch using KiCAD
Creating drawing-size SVG and PDF images of kicad/A1/DIN_A_SPICE.kicad_sch
Plotted to '/home/anton/DATA/www/SEDwebsite/_build/SEDwebsite/EE4109-2025-2026/designExample/HearingLoop/Notebooks/img/DIN_A_SPICE.svg'.
Done.


In [20]:
sl.makeCircuit("kicad/A1/DIN_A_SPICE_TEST.kicad_sch", language="SPICE")
sl.img2html("DIN_A_SPICE_TEST.svg", 400)

Creating netlist of kicad/A1/DIN_A_SPICE_TEST.kicad_sch using KiCAD
Creating drawing-size SVG and PDF images of kicad/A1/DIN_A_SPICE_TEST.kicad_sch
Plotted to '/home/anton/DATA/www/SEDwebsite/_build/SEDwebsite/EE4109-2025-2026/designExample/HearingLoop/Notebooks/img/DIN_A_SPICE_TEST.svg'.
Done.


In [19]:
sim_cmd = "AC DEC 20 20 20k"
names   = {"V_out": "V(out)"}
mag, phase, x_name, x_units  = sl.ngspice2traces("cir/DIN_A_SPICE_TEST", sim_cmd, names, traceType='dBmagPhase')
sl.plot("DIN_A_SPICE", "DIN A weighting function", "semilogx", mag, xName=x_name, xUnits=x_units)
sl.img2html("DIN_A_SPICE.svg", 400)